In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna>=3.6", "catboost>=1.2"])
sys.path.insert(0, "/kaggle/input/revway-forecasting-src/ml")

import pyarrow.parquet as pq
from models.forecasting.data import categorical_feature_names, prepare_xy
from models.forecasting.splits import hotel_wise_split
from models.forecasting.optuna_study import run_study

df = pq.read_table("/kaggle/input/revway-sample-5m/sample_5M_seed42.parquet").to_pandas()
X, y, _ = prepare_xy(df)
idx = hotel_wise_split(df["hotel_name_normalized"], seed=42)
study = run_study("xgboost", X, y, idx, categorical_feature_names(),
                  n_trials=50, storage="sqlite:////kaggle/working/study_xgboost.db",
                  study_name="bakeoff_xgboost", device="cuda")
print("BEST", study.best_value, study.best_params)
